In [ ]:
import pandas as pd

# CSV load karo
df = pd.read_excel("transcriptions_final.xlsx")   # apni file ka naam likho

# Emotions list
emotions = ["anger", "happy", "sad", "love", "neutral"]

# Function jo link se label nikale
def extract_label(link):
    link = link.lower()  # small letters mein convert
    for emotion in emotions:
        if emotion in link:
            return emotion
    return "unknown"  # agar koi emotion na mile

# Naya label column create karo
df["Label"] = df["Link"].apply(extract_label)   # "link" column ka naam check kar lena

# Check karo
print(df[["Link", "Label"]].head())

# Optional: save new CSV
df.to_csv("labeled_file.csv", index=False)

                                                Link  Label
0  /content/drive/MyDrive/Colab Notebooks/Thesis/...  anger
1  /content/drive/MyDrive/Colab Notebooks/Thesis/...  anger
2  /content/drive/MyDrive/Colab Notebooks/Thesis/...  anger
3  /content/drive/MyDrive/Colab Notebooks/Thesis/...  anger
4  /content/drive/MyDrive/Colab Notebooks/Thesis/...  anger


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
!pip uninstall transformers -y
!pip install transformers datasets evaluate accelerate -U

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)


In [ ]:
import transformers
print(transformers.__version__)

5.2.0


In [ ]:
# ------------------- XLM-R Fine-Tuning for Urdu Text -------------------
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
import torch

# ------------------- Load Data -------------------
input_file = "labeled_file.csv"
df = pd.read_csv(input_file)

# Keep only relevant columns
df = df[['text', 'Label']]

# Encode labels
labels = list(set(df['Label']))
label_encoding = {label: idx for idx, label in enumerate(labels)}
rev_label_encoding = {v: k for k, v in label_encoding.items()}
df['Label'] = df['Label'].map(label_encoding)

# ------------------- Train/Test Split -------------------
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
data = DatasetDict({'train': train_dataset, 'test': test_dataset})

# ------------------- Tokenizer -------------------
model_name = "xlm-roberta-base"  # or "xlm-roberta-base" if GPU memory is small
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = data.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("Label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
# ------------------- Model -------------------
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=rev_label_encoding,
    label2id=label_encoding,
    ignore_mismatched_sizes=True
)

# ------------------- Metrics -------------------
metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'f1': metric_f1.compute(predictions=predictions, references=labels, average='macro')['f1'],
        'accuracy': metric_acc.compute(predictions=predictions, references=labels)['accuracy']
    }

# ------------------- Training Arguments -------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    learning_rate=5e-5,
    logging_dir='./logs',
    logging_strategy='steps',
    logging_steps=50
)

# ------------------- Trainer -------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics
)

# ------------------- Train & Evaluate -------------------
trainer.train()
trainer.evaluate()

# ------------------- Save Model -------------------
model.save_pretrained('./fine_tuned_xlm_roberta')
tokenizer.save_pretrained('./fine_tuned_xlm_roberta')
print("Fine-tuned XLM-R model saved successfully!")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/6635 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned XLM-R model saved successfully!


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# Load the pre-trained XLM-RoBERTa model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")
model = AutoModel.from_pretrained("xlm-roberta-large")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm # For a nice progress bar

# 1. Configuration
BATCH_SIZE = 16  # Adjust based on your GPU memory (use 8 for 'large' models)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. Optimized function for batching
def get_batch_embeddings(batch_texts):
    # Tokenize the entire batch at once
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.base_model(**inputs) # Access the base transformer layer
        # Mean Pooling: Average the token embeddings to get sentence embeddings
        embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.cpu().numpy()

# 3. Process in loops of BATCH_SIZE
all_embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i : i + BATCH_SIZE]
    batch_vectors = get_batch_embeddings(batch)
    all_embeddings.extend(batch_vectors)

# 4. Save using Pickle (Preserves the math structure better than CSV)
embeddings_df = pd.DataFrame(all_embeddings)
embeddings_df['Link'] = links
embeddings_df['Label'] = labels

embeddings_df.to_csv('urdu_embeddings_optimized.csv')
print("Finished! Saved as a csv file.")

100%|██████████| 519/519 [00:24<00:00, 21.46it/s]


Finished! Saved as a csv file.


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


# Load the features from the Excel file
input_file = ("urdu_embeddings_optimized.csv")
df = pd.read_csv(input_file)
df = df.drop(columns=["Link"])

In [14]:
from sklearn.preprocessing import LabelEncoder
# Split the data into features (X) and labels (y)
#df = df.drop(['Link'],axis = 1)
data = df.iloc[::]  # Features
Y = df['Label']      # Target variable
X = data.drop(['Label'],axis = 1)

label_encoder = LabelEncoder()
Y = label_encoder.fit_transform(Y)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [15]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

# Define and train the XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=100,      # Number of boosting rounds (trees)
    tree_method="hist",
    learning_rate=0.5,     # Learning rate for boosting
    max_depth=6,           # Maximum depth of trees
    eval_metric='mlogloss', # Log loss for multi-class classification
    use_label_encoder=False # Disable automatic label encoding
)

# Train the XGBoost classifier on the extracted features and corresponding labels
xgb_model.fit(X_train, Y_train)

y_pred = xgb_model.predict(X_test)

# Calculate test accuracy
accuracy = accuracy_score(Y_test, y_pred)
print(f"XGBoost Test Accuracy: {accuracy * 100:.2f}%")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:02:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Test Accuracy: 60.58%
